2. Data Cleaning

In [6]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

import pandas as pd
RAW, OUT, REPORT = ROOT/'data/raw', ROOT/'data/processed', ROOT/'results/reports'
OUT.mkdir(parents=True, exist_ok=True); REPORT.mkdir(parents=True, exist_ok=True)
forms = pd.read_csv(RAW/'Construction_Data_PM_Forms_All_Projects.csv', low_memory=False).drop_duplicates().copy()
tasks = pd.read_csv(RAW/'Construction_Data_PM_Tasks_All_Projects.csv', low_memory=False).drop_duplicates().copy()

def clean(df):
    # 1. makeing column names standard
    df.columns = (df.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
    
    # 2. converting data columns to dates
    for c in [c for c in df.columns if c in {'created', 'status_changed'}]:
        df[c] = pd.to_datetime(df[c], dayfirst=True, errors='coerce')
        
    # 3. converting target column to date ( from int)
    if 'target' in df.columns:
        # 1899-12-30 is the start date for excel and we need to add the number to this start date to get the exact date
        df['target'] = pd.to_datetime(
            pd.to_numeric(df['target'], errors='coerce'), 
            unit='D', 
            origin='1899-12-30', 
            errors='coerce'
        )
        
    # 4. making string columns similar
    for c in df.select_dtypes(include='object'):
        df[c] = df[c].astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA})
        
    return df

forms, tasks = clean(forms), clean(tasks)
# Preserve missingness in sparse categorical fields instead of inventing a substantive value.
for df in (forms, tasks):
    for c in df.select_dtypes(include='string'):
        df[c] = df[c].fillna('Unknown')
forms.to_csv(OUT/'forms_cleaned.csv', index=False)
tasks.to_csv(OUT/'tasks_cleaned.csv', index=False)
pd.DataFrame({'forms_rows':[len(forms)], 'tasks_rows':[len(tasks)], 'forms_missing':[int(forms.isna().sum().sum())], 'tasks_missing':[int(tasks.isna().sum().sum())]}).to_csv(REPORT/'cleaning_audit.csv', index=False)


In this step I need to understand the outliers. I won't delete any of them in this step since they must be understood first. If some of them seem to be wrong or false we can delete them. 

In [8]:
numeric = tasks.select_dtypes('number')
q = numeric.quantile([.25,.75]); iqr = q.loc[.75]-q.loc[.25]
outliers = ((numeric < (q.loc[.25]-1.5*iqr)) | (numeric > (q.loc[.75]+1.5*iqr))).sum()
outliers.to_csv(REPORT/'task_outlier_counts.csv'); outliers


project    0
dtype: int64

In [9]:
numeric = forms.select_dtypes('number')
q = numeric.quantile([.25,.75]); iqr = q.loc[.75]-q.loc[.25]
outliers = ((numeric < (q.loc[.25]-1.5*iqr)) | (numeric > (q.loc[.75]+1.5*iqr))).sum()
outliers.to_csv(REPORT/'form_outlier_counts.csv'); outliers

open_actions      204
total_actions    1482
project             0
dtype: int64